In [11]:
import re
import subprocess
from pathlib import Path
import pymupdf
import pymupdf4llm
from shared import embedder


In [12]:
import re
import subprocess
from pathlib import Path


from shared import embedder

DOCS_DIR = Path("../documents")


In [13]:
def ensure_search_indexes(collection) -> None:
    existing = {index["name"] for index in collection.list_search_indexes()}
    created = []
    collection.create_search_index(
        {
            "name": "knowledge_vector_index",
            "type": "vectorSearch",
            "definition": {
                "fields": [
                    {
                        "type": "vector",
                        "path": "embedding",
                        "numDimensions": EMBEDDING_DIMENSION,
                        "similarity": "cosine",
                    }
                ]
            },
        }
    )
    collection.create_search_index(
        {
            "name": "knowledge_text_index",
            "definition": {
                "mappings": {
                    "dynamic": False,
                    "fields": {
                        "text": {"type": "string", "analyzer": "lucene.russian"},
                        "section_title": {"type": "string", "analyzer": "lucene.russian"},
                    },
                }
            },
        }
    )
    print(f"Waiting for search indexes to build: {created}")
    deadline = tm.monotonic() + 180
    while tm.monotonic() < deadline:
        statuses = {index["name"]: index for index in collection.list_search_indexes()}
        if all(statuses.get(name, {}).get("queryable") for name in created):
            print("Search indexes are queryable.")
            return
        tm.sleep(2)
    raise TimeoutError(f"Search indexes did not become queryable in time: {created}")

In [14]:
import hashlib
import json
import os
import re
import time as tm
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pymongo
import pymongo.errors
import pymupdf4llm
from langchain_text_splitters import RecursiveCharacterTextSplitter
from shared import embedder

# Must match backend/src/storages/mongo/knowledge.py's KnowledgeChunk.Settings —
# duplicated, not imported, since backend/ingest deliberately doesn't depend on
# the backend package (see [[backend-ingest-separate-project]]).
COLLECTION_NAME = "knowledge_chunks"
VECTOR_INDEX_NAME = "knowledge_vector_index"
TEXT_INDEX_NAME = "knowledge_text_index"
EMBEDDING_DIMENSION = 1024  # mxbai-embed-large
MONGO_URI =  "mongodb://localhost/db?directConnection=true"


client = pymongo.MongoClient(MONGO_URI)
collection = client.get_database()[COLLECTION_NAME]

In [15]:
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
# mxbai-embed-large's Ollama context window is 512 tokens; Cyrillic text runsq
# well under 1 char/token, so keep chunks well short of that to avoid MV_HTTP 500s.
CHUNK_SIZE = 400
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40)

document_path = Path("./documents/Инструкция_по_работе_с_машиночитаемыми_доверенностями.pdf")


page_count = pymupdf.open(document_path).page_count
doc = pymupdf4llm.to_markdown(document_path, pages=list(range(2, page_count)))





In [41]:
document_path

PosixPath('documents/Инструкция_по_работе_с_машиночитаемыми_доверенностями.pdf')

In [78]:
from pprint import pprint
from uuid import uuid4
splitted = splitter.split_text(doc)


items = [
                    {
                    "id": str(uuid4()),
                    "text": item,
                    "path": str(document_path),
                    "label": "manual",
                    "term": None,
                }
                for item in splitted
]

In [88]:
def embed_and_insert_batch(collection, batch: list[dict]) -> tuple[int, int]:
    """Embed one batch and save it immediately, so progress lands in Mongo as
    each batch finishes rather than only after every batch has embedded."""
    embeddings = embedder.embed_documents([item["text"] for item in batch])
    for item, embedding in zip(batch, embeddings, strict=True):
        item["embedding"] = [float(x) for x in embedding]


    result = collection.insert_many(batch, ordered=False)
    return len(result.inserted_ids), 0

KeyError: '_id'

In [103]:
# embed_and_insert_batch(collection, items)
result = collection.insert_many(items)

In [104]:

def embed_and_insert_in_pool(data):
    BATCH_SIZE = 32
    MAX_WORKERS = 6

    batches = [data[i : i + BATCH_SIZE] for i in range(0, len(data), BATCH_SIZE)]
    inserted_total = 0
    duplicate_total = 0
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = [pool.submit(embed_and_insert_batch, collection, batch) for batch in batches]
        for done, future in enumerate(as_completed(futures), start=1):
            inserted, duplicates = future.result()
            inserted_total += inserted
            duplicate_total += duplicates
            print(f"Batch {done}/{len(batches)} done ({inserted} inserted, {duplicates} duplicate text skipped)")


In [19]:
query = 'определении ИНН'
query_vector = embedder.embed_query(query)

vector_pipeline = [
    {
        "$vectorSearch": {
            "index": VECTOR_INDEX_NAME,
            "path": "embedding",
            "queryVector": query_vector,
            "numCandidates": 20,
            "limit": 20,
        }
    },
    {"$project": {"embedding": 0}},
]
text_pipeline = [
    {"$search": {"index": TEXT_INDEX_NAME, "text": {"query": query, "path": ["text", "section_title"]}}},
    {"$limit": 20},
    {"$project": {"embedding": 0}},
]
vector_hist = collection.aggregate(vector_pipeline).to_list(length=20)
text_hits = collection.aggregate(text_pipeline).to_list(length=20)


[{'_id': ObjectId('6aa5b8e9aba79fd8d4b4f27d'),
  'id': 'd40faa84-dfbd-456b-bca0-8dc270ee7722',
  'text': '- Начальная цена котировочной сессии; \n\n- Дата начала котировочной сессии; \n\n- Дата окончания котировочной сессии; \n\n- Наименование заказчика; \n\n- ИНН заказчика; \n\n- Законы, в соответствии с которыми осуществляется/осуществлялась закупка; \n\n- Основание заключения контракта; \n\n- Способ размещения закупки (заказа)/определение поставщика.',
  'path': 'documents/Инструкция_по_работе_с_Порталом_для_заказчика.pdf',
  'label': 'manual',
  'term': None},
 {'_id': ObjectId('6aa5b889aba79fd8d4b4f1d0'),
  'id': 'f1576800-0465-4528-b018-803f865e0e01',
  'text': '− Выбрать параметры получения для различных типов уведомлений (уведомления на Портале поставщиков либо email); \n\n− Выполнить подписку на получение уведомлений определенного типа (установить флажок); \n\n− Отменить подписку на получение уведомлений определенного типа.',
  'path': 'documents/Инструкция_по_работе_с_Портало

In [6]:
hints = vector_hist + text_hits
len(hints)

40

In [7]:
documents = [
    hit['text'] for hit in hints
]

In [10]:
documents[32]

'При нажатии на кнопку «Операции с МЧД» откроется модальное окно «Машиночитаемые доверенности пользователя» (Рисунок ), в котором администратору необходимо загрузить доверенность, добавив xml-файл в поле «Загрузить МЧД из файла» (Рисунок  (1)) и нажать на кнопку «Добавить» (Рисунок  (2)). \n\n\n\n**Рисунок 6 – Модальное окно «Машиночитаемые доверенности пользователя»**'

In [8]:
from sentence_transformers import CrossEncoder

reranker_model = CrossEncoder('DiTy/cross-encoder-russian-msmarco', max_length=512)


rank_result = reranker_model.rank(query, documents)
print(rank_result)
# `[{'corpus_id': 0, 'score': 0.88126713},
#  {'corpus_id': 2, 'score': 0.001042091},
#  {'corpus_id': 3, 'score': 0.0010417715},
#  {'corpus_id': 1, 'score': 0.0010344835},
#  {'corpus_id': 4, 'score': 0.0010244923}]`


'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/DiTy/cross-encoder-russian-msmarco/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/DiTy/cross-encoder-russian-msmarco/resolve/main/./modules.json
Retrying in 2s [Retry 2/5].
'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/DiTy/cross-encoder-russian-msmarco/resolve/main/./modules.json
Retrying in 4s [Retry 3/5].
'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/DiTy/cross-encoder-russian-msmarco/resolve/main/./modules.json
Retrying in 8s [Retry 4/5].
'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/DiTy/cross-encoder-russian-msmarco/resolve/main/./modules.json
Retrying in 8s [Retry 5/5].
'[Errno -3] Temporary failure in na

[{'corpus_id': 32, 'score': 0.5695022344589233}, {'corpus_id': 5, 'score': 0.5100562572479248}, {'corpus_id': 22, 'score': 0.5100562572479248}, {'corpus_id': 12, 'score': 0.4832008183002472}, {'corpus_id': 29, 'score': 0.4832008183002472}, {'corpus_id': 0, 'score': 0.4391607642173767}, {'corpus_id': 23, 'score': 0.4391607642173767}, {'corpus_id': 13, 'score': 0.3190537691116333}, {'corpus_id': 24, 'score': 0.3190537691116333}, {'corpus_id': 10, 'score': 0.2179134488105774}, {'corpus_id': 26, 'score': 0.2179134488105774}, {'corpus_id': 4, 'score': 0.18631470203399658}, {'corpus_id': 21, 'score': 0.18631470203399658}, {'corpus_id': 2, 'score': 0.17089588940143585}, {'corpus_id': 20, 'score': 0.17089588940143585}, {'corpus_id': 27, 'score': 0.12966132164001465}, {'corpus_id': 17, 'score': 0.12813059985637665}, {'corpus_id': 25, 'score': 0.12813059985637665}, {'corpus_id': 3, 'score': 0.05208312347531319}, {'corpus_id': 28, 'score': 0.05208312347531319}, {'corpus_id': 34, 'score': 0.051384

In [ ]:
rank_result[:5]

[{'corpus_id': 32, 'score': 0.5695022344589233},
 {'corpus_id': 5, 'score': 0.5100562572479248},
 {'corpus_id': 22, 'score': 0.5100562572479248},
 {'corpus_id': 12, 'score': 0.4832008183002472},
 {'corpus_id': 29, 'score': 0.4832008183002472}]

In [63]:
results

[{'text': 'кнопку/значок»** или **«щелчком выбрать наименование/название»** .',
  'id': 'df19c2c4-493e-40af-94ef-8071d038d1e0',
  'score': np.float32(0.99940985)},
 {'text': 'автоматический отказ поставщика от контракта.</mark>',
  'id': '92b2ef8f-6e94-43d7-99e5-17d1b9336508',
  'score': np.float32(0.999394)},
 {'text': 'Для удаления электронной подписи, пользователю необходимо нажать на кнопку «Операции с ЭП» (Рисунок 113 (4)) и затем нажать на кнопку «Удалить электронную подпись» (Рисунок 117 (3)). \n\n### **4.1.5. Добавление МЧД**',
  'id': 'f43ab471-6d71-4647-9947-94e0cd2fcc7e',
  'score': np.float32(0.99928325)},
 {'text': '**Рисунок 341 – Редактирование документа заказчика** \n\n239 \n\n## **14.4.Удаление документа заказчика из электронной библиотеки документов** \n\nДля удаления документа заказчика необходимо в таблице реестра документов отметить его флажком и нажать на кнопку «Удалить» (Рисунок 342). \n\n\n\n**Рисунок 342 – Режим удаления документов поставщика в электронной биб

In [22]:
document_path = Path("./documents/Инструкция_по_работе_с_Порталом_для_заказчика.pdf")


page_count = pymupdf.open(document_path).page_count
doc = pymupdf4llm.to_markdown(document_path, pages=list(range(4, page_count - 1)))


In [106]:
splitted = splitter.split_text(doc)


items = [
                    {
                    "id": str(uuid4()),
                    "text": item,
                    "path": str(document_path),
                    "label": "manual",
                    "term": None,
                }
                for item in splitted
]

In [107]:
embed_and_insert_in_pool(items)

Batch 1/24 done (32 inserted, 0 duplicate text skipped)
Batch 2/24 done (32 inserted, 0 duplicate text skipped)
Batch 3/24 done (32 inserted, 0 duplicate text skipped)
Batch 4/24 done (32 inserted, 0 duplicate text skipped)
Batch 5/24 done (32 inserted, 0 duplicate text skipped)
Batch 6/24 done (32 inserted, 0 duplicate text skipped)
Batch 7/24 done (32 inserted, 0 duplicate text skipped)
Batch 8/24 done (32 inserted, 0 duplicate text skipped)
Batch 9/24 done (32 inserted, 0 duplicate text skipped)
Batch 10/24 done (32 inserted, 0 duplicate text skipped)
Batch 11/24 done (32 inserted, 0 duplicate text skipped)
Batch 12/24 done (32 inserted, 0 duplicate text skipped)
Batch 13/24 done (32 inserted, 0 duplicate text skipped)
Batch 14/24 done (32 inserted, 0 duplicate text skipped)
Batch 15/24 done (32 inserted, 0 duplicate text skipped)
Batch 16/24 done (32 inserted, 0 duplicate text skipped)
Batch 17/24 done (32 inserted, 0 duplicate text skipped)
Batch 18/24 done (32 inserted, 0 duplica

In [24]:
# document_path = Path("./documents/Инструкция_по_работе_с_Порталом_для_поставщика.pdf")
document_path = Path("./documents/Инструкция_по_работе_с_Порталом_для_заказчика.pdf")

page_count = pymupdf.open(document_path).page_count
json_string = pymupdf4llm.to_json(document_path, pages=[3])


In [23]:
from pprint import pprint
import json


def extract_table_glossary(data, glossary_source):
    extract = []
    for page in data.get("pages", []):
        for block in page.get("boxes", []):
            if block["boxclass"] == "table":
                extract.extend(block["table"]["extract"][1:])  # skip header row


    term_items = [
    {
        "title": term.strip(),
        "text": f"{term.strip()} — {definition.strip()}",
        "label": "glossary",
        "labels": ["glossary"],
        "metadata": {
            "source": glossary_source.name[:-4],
            "document": glossary_source.name[:-4],
            "path": str(glossary_source),
            "term": term.strip(),
        },
    }
    for term, definition in extract
    if term.strip() and definition.strip()
    ]
    return term_items



In [18]:
term_items = extract_table_glossary(json.loads(json_string), document_path)

embed(term_items)

In [19]:
doc = pymupdf4llm.to_markdown(document_path, pages=list(range(4, page_count)))


In [20]:
objs = split_sections(doc, document_path.name[:-4])


In [21]:
objs[2:] = objs

In [22]:
embeddings = [[float(x) for x in vec] for vec in embedder.embed_documents([item["text"] for item in objs])]


In [23]:
for item, embedding in zip(objs, embeddings):
    mem.put_many([item], embeddings=[embedding])


In [31]:
mem.find("физическое лицо", k=100)

{'query': 'физическое лицо',
 'hits': [{'frame_id': 214,
   'uri': 'mv2://frames/214',
   'title': '54 - **Рисунок 80 – Страница с системным сообщение об отправке ссылки-подтверждения на адрес электронной почты** (part 2)',
   'rank': 1,
   'score': 10.75866413116455,
   'matches': 3,
   'snippet': '- 5 На форме « **Выбор типа учетной записи** » выбрать один из вариантов ниже и нажать на кнопку «Продолжить» (Рисунок 82): - зарегистрировать юридическое лицо; - зарегистрировать индивидуального предпринимателя; - зарегистрировать физическое лицо. > 5 В случае, если письмо со ссылкой подтверждения не пришло, то можно запросить его повторно по кнопке «Нажмите сюда» title: 54 - **Рисунок 80 – Страница с системным сообщение об отправке ссылки-подтверждения на адрес электронной ...',
   'text': '- 5 На форме « **Выбор типа учетной записи** » выбрать один из вариантов ниже и нажать на кнопку «Продолжить» (Рисунок 82): - зарегистрировать юридическое лицо; - зарегистрировать индивидуального предп

In [27]:
document_path = Path("./documents/Инструкция_по_созданию_оферты_и_СТЕ.pdf")


page_count = pymupdf.open(document_path).page_count



In [28]:
json_string = pymupdf4llm.to_json(document_path, pages=[2])
term_items = extract_table_glossary(json.loads(json_string), document_path)

In [29]:
embed(term_items)

In [30]:
doc = pymupdf4llm.to_markdown(document_path, pages=list(range(3, page_count)))

In [26]:
doc

'# **Введение** \n\nНастоящая инструкция описывает процесс создания поставщиком оферт \n\nи СТЕ на Портале поставщиков. \n\nВ инструкции описаны следующие способы добавления оферт: \n\n- подача ценового предложения на основе СТЕ; \n\n- загрузка файлов в форматах YML и XLSX; \n\n- создание предложения и СТЕ через экранную форму; \n\n- создание оферт на основе КС. \n\nКроме того, в инструкции содержится подробное описание по работе с офертами, в том числе по офертам, созданным на основе КС. \n\n4 \n\n## **1. Подача ценового предложения** \n\nЕсли СТЕ имеет вариативность, то ценовое предложение будет создано только для того <mark>варианта СТЕ, с которого была нажата кнопка «Опубликовать предложение».</mark> Если необходимая СТЕ уже размещена в Каталоге ТРУ, поставщику достаточно \n\nсоздать ценовое предложение на ее основе. Для этого в Каталоге ТРУ выберите нужный товар, перейдите в его карточку, нажмите «Действия с СТЕ и офертами» и в открывшемся списке выберите «Опубликовать предложение

In [31]:
objs = split_sections(doc, document_path.name[:-4])
embeddings = [[float(x) for x in vec] for vec in embedder.embed_documents([item["text"] for item in objs])]

In [32]:
for item, embedding in zip(objs, embeddings):
    mem.put_many([item], embeddings=[embedding])


In [33]:
document_path = Path("./documents/Инструкция_по_формированию_YML.pdf")


page_count = pymupdf.open(document_path).page_count
doc = pymupdf4llm.to_markdown(document_path, pages=list(range(3, page_count)))


In [34]:
objs = split_sections(doc, document_path.name[:-4])
embeddings = [[float(x) for x in vec] for vec in embedder.embed_documents([item["text"] for item in objs])]

In [35]:
for item, embedding in zip(objs, embeddings):
    mem.put_many([item], embeddings=[embedding])


In [38]:

STORE_PATH = Path("ingest.ipynb").absolute().parent.parent / 'data' / 'knowledge-third.mv2'
STORE_PATH
mem = use(
    "basic",
    str(STORE_PATH.absolute()),
    enable_lex=True,
    enable_vec=True,
)

In [42]:
document_path = Path("./documents/Инструкция_по_электронному_актированию.pdf")


page_count = pymupdf.open(document_path).page_count
json_string = pymupdf4llm.to_json(document_path, pages=[3])

extract = []
data = json.loads(json_string)
for page in data.get("pages", []):
    for block in page.get("boxes", []):
        if block["boxclass"] == "table":
            extract.extend(block["table"]["extract"][1:])  # skip header row
glossary_source = document_path

term_items = [
{
    "title": term.strip(),
    "text": f"{term.strip()} — {definition.strip()}",
    "label": "glossary",
    "labels": ["glossary"],
    "metadata": {
        "source": glossary_source.name[:-4],
        "document": glossary_source.name[:-4],
        "path": str(glossary_source),
        "term": term.strip(),
    },
}
for _, term, definition in extract
if term.strip() and definition.strip()
]


In [44]:
embed(term_items)

In [46]:
json_string = pymupdf4llm.to_markdown(document_path, pages=list(range(4, page_count)))

In [47]:
doc = json_string

In [48]:
doc

'5 \n\n# **2 Аннотация** \n\nДокумент предназначен для описания действий пользователя Поставщика на Портале поставщиков при работе с электронным исполнением контрактов через ЕИС и Калуга Астрал в качестве оператора ЭДО. \n\nДокумент содержит: \n\n- Обучающие видеоролики и материалы вебинаров по ЭИ через ПП; \n\n- Нормативные акты по ЭИ; \n\n- Обязательные шаги, которые необходимо выполнить поставщику перед началом выставления документов электронного актирования на Портале поставщиков; \n\n- Инструкцию по созданию исполнений и заполнению полей; \n\n- Схемы сценариев работы и перечень доступных действий поставщика; \n\n- Разбор нестандартных случаев и часто повторяющихся вопросов. \n\n6 \n\n# **3 Видеоролики и материалы вебинаров** \n\nДоступны ознакомительные видеоролики, записи вебинаров и материалы вебинаров (Таблица 2): \n\n**Таблица 2 – Ознакомительные видеоролики** \n\n|**№**|**Описание**|**Ссылка**|\n|---|---|---|\n|1)|Видео о работе в интерфейсе с текстовыми<br>комментариями и оз

In [49]:
objs = split_sections(doc, document_path.name[:-4])
embeddings = [[float(x) for x in vec] for vec in embedder.embed_documents([item["text"] for item in objs])]

In [ ]:

STORE_PATH = Path("ingest.ipynb").absolute().parent.parent / 'data' / 'knowledge-third.mv2'
STORE_PATH
mem = use(
    "basic",
    str(STORE_PATH.absolute()),
    enable_lex=True,
    enable_vec=True,
)

In [53]:
for item, embedding in zip(objs, embeddings):
    mem.put_many([item], embeddings=[embedding])


In [ ]:
mem.find("Модальное окно с выбором сведений из существующего исполнения", mode="lex", embedder=embedder)

{'query': 'Модальное окно с выбором сведений из существующего исполнения',
 'hits': [{'frame_id': 678,
   'uri': 'mv2://frames/678',
   'title': '211 ### **Рисунок 352 – Этап контракта, блок «Исполнения для грузополучателей»** (part 1)',
   'rank': 1,
   'score': 29.53182029724121,
   'matches': 23,
   'snippet': '**Рисунок 353 – Окно подтверждения создания исполнения для грузополучателя** **Рисунок 354 – Модальное окно с выбором сведений из существующего исполнения** title: 211 ### **Рисунок 352 – Этап контракта, блок «Исполнения для грузополучателей»** (part 1) labels: manual section: "211" section_title: "### **Рисунок 352 – Этап контракта, блок «Исполнения для грузополучателей»**" source: "Инструкция_по_работе_с_Порталом_для_поставщика"',
   'text': '**Рисунок 353 – Окно подтверждения создания исполнения для грузополучателя** **Рисунок 354 – Модальное окно с выбором сведений из существующего исполнения** title: 211 ### **Рисунок 352 – Этап контракта, блок «Исполнения для грузополуч